# 04 — Evaluation: Thresholds, Subgroups and Calibration

**Project:** 30-Day Readmission Risk in Diabetic Inpatients
**Notebook 4 of 5**



---
## 0. Setup

In [1]:
# Imports and display settings

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import (confusion_matrix, precision_score, recall_score,
                             f1_score, roc_auc_score, average_precision_score,
                             brier_score_loss)
from sklearn.calibration import calibration_curve
from statsmodels.stats.proportion import proportion_confint

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 200)
sns.set_theme(style="whitegrid")

DATA_DIR = "../data"

### Load the test-set predictions from notebook 03

Notebook 03 saved the held-out test set with the model's predicted probabilities
attached. Everything here works from that file, so nothing is re-fitted and there
is no risk of accidentally evaluating on training data.

In [2]:
# Read the predictions notebook 03 saved

preds = pd.read_csv(f"{DATA_DIR}/processed/test_predictions.csv")

print(f"Test-set encounters: {len(preds):,}")
print(f"Patients:            {preds['patient_nbr'].nunique():,}")
print(f"Observed base rate:  {preds['target'].mean():.1%}")
print(f"\nColumns: {list(preds.columns)}")

Test-set encounters: 19,634
Patients:            13,996
Observed base rate:  11.1%

Columns: ['encounter_id', 'patient_nbr', 'target', 'age', 'race', 'gender', 'under_20', 'payer_code', 'prob']


In [3]:
# Sanity check against notebook 03

assert len(preds) == 19634, f"Expected 19,634 test encounters, got {len(preds):,}"
assert "prob" in preds.columns, "predicted probabilities missing"
assert abs(preds["target"].mean() - 0.111) < 0.005, "base rate does not match notebook 03"

y = preds["target"].values
p = preds["prob"].values
print("Matches notebook 03.")

Matches notebook 03.


---
## 1. The threshold is a clinical decision, not a statistical one

A model outputs a probability. To act on it, someone has to draw a line: above
this number, the patient goes on the discharge team's review list; below it, they
don't. That line is the threshold, and where it sits is the single most
consequential choice in deploying this model.

The default is 0.5. It is almost always wrong, and here it is nonsensical — at an
11% base rate, almost nobody reaches a predicted probability of 0.5, so a
0.5 threshold flags almost no one and the tool does nothing.

The right threshold depends on two costs the model cannot know:

- **A false negative** — a patient flagged low-risk who is readmitted. The cost is
  a preventable readmission: a patient who deteriorates at home, a bed used again,
  and in a Kenyan setting a family paying twice.
- **A false positive** — a patient flagged high-risk who would not have been
  readmitted. The cost is a discharge review that turns out not to have been
  needed: a medication reconciliation, an educator session, a phone call.

These costs are wildly asymmetric. A missed readmission is a clinical harm. A
wasted review is a cheap, low-risk, often mildly beneficial intervention given to
someone who didn't strictly need it. So the threshold should be set low —
tolerating many false positives to avoid false negatives — but not so low that the
list is longer than the team can work through, at which point the tool stops
helping.

---
## 2. What each threshold would actually do

Rather than optimise an abstract metric, this walks through candidate thresholds
and shows what each one means operationally: how many patients get flagged, how
many of them are genuine, and how many readmissions are missed.

In [4]:
# For a range of thresholds, show the operational consequences

def threshold_row(thresh):
    flag = (p >= thresh).astype(int)
    tp = int(((flag == 1) & (y == 1)).sum())
    fp = int(((flag == 1) & (y == 0)).sum())
    fn = int(((flag == 0) & (y == 1)).sum())
    tn = int(((flag == 0) & (y == 0)).sum())
    flagged = tp + fp
    return {
        "threshold": thresh,
        "flagged": flagged,
        "flagged_pct": round(100 * flagged / len(y), 1),
        "caught": tp,
        "missed": fn,
        "recall": round(100 * tp / (tp + fn), 1),
        "precision": round(100 * tp / flagged, 1) if flagged else 0.0,
    }

rows = [threshold_row(t) for t in [0.08, 0.10, 0.12, 0.15, 0.20, 0.25, 0.30, 0.40, 0.50]]
table = pd.DataFrame(rows)
print(f"Total readmissions in test set: {int(y.sum())}")
print(f"Total encounters: {len(y):,}\n")
print(table.to_string(index=False))

Total readmissions in test set: 2179
Total encounters: 19,634

 threshold  flagged  flagged_pct  caught  missed  recall  precision
      0.08    13090         66.7    1799     380    82.6       13.7
      0.10     8944         45.6    1432     747    65.7       16.0
      0.12     6065         30.9    1082    1097    49.7       17.8
      0.15     3490         17.8     750    1429    34.4       21.5
      0.20     1726          8.8     449    1730    20.6       26.0
      0.25      964          4.9     296    1883    13.6       30.7
      0.30      552          2.8     192    1987     8.8       34.8
      0.40      201          1.0      86    2093     3.9       42.8
      0.50       82          0.4      41    2138     1.9       50.0


In [5]:
# Cell 2b — locate the statistical optimum so the chosen threshold reads as a decision, not an eyeball
from sklearn.metrics import precision_recall_curve
import numpy as np

prec, rec, thr = precision_recall_curve(y, p)

# precision_recall_curve returns len(thr) = len(prec) - 1; drop the trailing point
prec_t, rec_t = prec[:-1], rec[:-1]

f1 = np.divide(2 * prec_t * rec_t, prec_t + rec_t,
               out=np.zeros_like(prec_t), where=(prec_t + rec_t) > 0)

best = int(np.argmax(f1))
even = int(np.argmin(np.abs(prec_t - rec_t)))

print(f"Max F1           : threshold {thr[best]:.3f} | precision {prec_t[best]*100:.1f}% | recall {rec_t[best]*100:.1f}% | F1 {f1[best]:.3f}")
print(f"P/R break-even   : threshold {thr[even]:.3f} | precision {prec_t[even]*100:.1f}% | recall {rec_t[even]*100:.1f}%")
print(f"Chosen (operating): threshold 0.125")

Max F1           : threshold 0.125 | precision 18.7% | recall 46.8% | F1 0.267
P/R break-even   : threshold 0.181 | precision 24.4% | recall 24.4%
Chosen (operating): threshold 0.125


In [6]:
# Cell 2c — bootstrap confidence intervals at the chosen operating threshold
import numpy as np

CHOSEN_THRESHOLD = 0.125
N_BOOT = 1000
rng = np.random.default_rng(42)

y_true_arr = np.asarray(y)
y_prob_arr = np.asarray(p)
n = len(y_true_arr)

boot_prec, boot_rec, boot_flag = [], [], []

for _ in range(N_BOOT):
    idx = rng.integers(0, n, n)
    yt, yp = y_true_arr[idx], y_prob_arr[idx]
    flagged = yp >= CHOSEN_THRESHOLD
    tp = int((flagged & (yt == 1)).sum())
    boot_prec.append(tp / flagged.sum() if flagged.sum() else np.nan)
    boot_rec.append(tp / (yt == 1).sum() if (yt == 1).sum() else np.nan)
    boot_flag.append(flagged.mean())

def ci(vals):
    return np.nanpercentile(vals, 2.5), np.nanpercentile(vals, 97.5)

for label, vals in [("Precision", boot_prec), ("Recall", boot_rec), ("Flagged %", boot_flag)]:
    lo, hi = ci(vals)
    print(f"{label:10s} at {CHOSEN_THRESHOLD}: {np.nanmean(vals)*100:5.1f}%  [95% CI {lo*100:.1f} - {hi*100:.1f}]")

Precision  at 0.125:  18.5%  [95% CI 17.5 - 19.6]
Recall     at 0.125:  46.8%  [95% CI 44.6 - 49.0]
Flagged %  at 0.125:  27.9%  [95% CI 27.3 - 28.5]


In [7]:
# Cell 2d — recall-weighted thresholds: F-beta sweep because a missed readmission costs more than a wasted flag
import numpy as np
from sklearn.metrics import precision_recall_curve

prec, rec, thr = precision_recall_curve(y, p)
prec_t, rec_t = prec[:-1], rec[:-1]

def fbeta_curve(precision, recall, beta):
    b2 = beta ** 2
    num = (1 + b2) * precision * recall
    den = (b2 * precision) + recall
    return np.divide(num, den, out=np.zeros_like(num), where=den > 0)

print(f"{'beta':>5} {'recall weight':>14} {'threshold':>10} {'precision':>10} {'recall':>8} {'flagged':>9}")
print("-" * 60)

for beta in [1, 2, 3]:
    fb = fbeta_curve(prec_t, rec_t, beta)
    i = int(np.argmax(fb))
    t = thr[i]
    flagged_pct = (p >= t).mean() * 100
    print(f"{beta:>5} {beta**2:>13}x {t:>10.3f} {prec_t[i]*100:>9.1f}% {rec_t[i]*100:>7.1f}% {flagged_pct:>8.1f}%")

 beta  recall weight  threshold  precision   recall   flagged
------------------------------------------------------------
    1             1x      0.125      18.7%    46.8%     27.8%
    2             4x      0.084      14.1%    80.0%     62.7%
    3             9x      0.062      12.2%    94.7%     86.4%


In [8]:
# Set the threshold chosen above. Change this value to match your decision.
CHOSEN_THRESHOLD = 0.125

flag = (p >= CHOSEN_THRESHOLD).astype(int)
tp = int(((flag == 1) & (y == 1)).sum())
fp = int(((flag == 1) & (y == 0)).sum())
fn = int(((flag == 0) & (y == 1)).sum())
tn = int(((flag == 0) & (y == 0)).sum())

print(f"At threshold {CHOSEN_THRESHOLD}:")
print(f"  Flagged for review:   {tp + fp:,} of {len(y):,} ({100*(tp+fp)/len(y):.1f}%)")
print(f"  Genuine (precision):  {100*tp/(tp+fp):.1f}%")
print(f"  Readmissions caught:  {tp} of {tp+fn} ({100*tp/(tp+fn):.1f}%)")
print(f"  Readmissions missed:  {fn}")
print(f"\nConfusion matrix:")
print(pd.DataFrame([[tn, fp], [fn, tp]],
                   index=["actually no", "actually yes"],
                   columns=["flagged no", "flagged yes"]).to_string())

At threshold 0.125:
  Flagged for review:   5,486 of 19,634 (27.9%)
  Genuine (precision):  18.6%
  Readmissions caught:  1019 of 2179 (46.8%)
  Readmissions missed:  1160

Confusion matrix:
              flagged no  flagged yes
actually no        12988         4467
actually yes        1160         1019


### Choosing an operating threshold

The model gives me a probability. A ward needs a yes or a no. That
conversion is mine to make, not the model's, and I want to be
honest that it is the most clinical decision in this whole project.

### What the trade actually looks like

Test set is 19,634 encounters with 2,179 readmissions, a base rate
of 11.1%. Reading down the threshold table, the two ends are both
useless in different ways. At 0.08 I catch 82.6% of readmissions
but flag 13,090 patients - two thirds of every discharge. No
discharge team is working through two thirds of the ward, so that
is not triage, it is a rubber stamp. At 0.50 precision hits 50%,
which sounds excellent until I notice it rests on 82 flagged
patients and catches 41 readmissions out of 2,179. Precise and
pointless.

One thing that did reassure me: at every threshold, precision
among the flagged is at least the threshold itself - 13.7% at
0.08, 21.5% at 0.15, 34.8% at 0.30. If the model were
systematically over-confident, precision would sit below that
line. It does not. That is an early calibration signal and I will
test it properly in section 3.

### Where the optimum sits

I did not want to pick a number by eye off nine hand-chosen rows,
so I searched every threshold the model produces. Max F1 lands at
0.125, precision-recall break-even at 0.181.

What struck me more than the optimum was how flat the curve is
around it. Going from 0.125 to 0.181 halves my recall, 46.8% down
to 24.4%, and precision only climbs from 18.7% to 24.4%. F1 barely
registers the difference. The data is not going to hand me a
threshold. Something outside the data has to decide it.

### Trying to encode the clinical asymmetry, and failing

My instinct was that a false negative costs more than a false
positive here, so I should weight recall. F-beta is the standard
way to say that, so I ran it.

It did not survive contact with this model. At beta=2 it wants me
to flag 62.7% of discharges; at beta=3, 86.4%, with precision down
to 12.2% against a base rate of 11.1%. At that point I have not
built a risk model, I have built something that says yes to almost
everybody.

The cause is the flatness above. Precision barely rises as I
tighten, so the only way F-beta can improve its score is to push
recall, and it runs to the floor. The deeper problem is that
F-beta weighs false negatives against false positives but has no
term for how many people I flagged. It assumes flags are free.
They are not - every flag is a discharge-planning slot, and a
wasted one came out of somebody else's.

So the clinical argument was right and the metric could not carry
it. I am keeping that rather than quietly picking beta=2 and
moving on.

### The threshold I am choosing, and what it commits me to

**0.125.** It is where F1 peaks, it flags 27.8% of discharges, and
it gets me 46.8% recall.

On capacity: I do not have a real service-level figure, and I am
not going to invent one to make the arithmetic look rigorous. What
I can say is that roughly one discharge in four is at the outer
edge of deliverable. A busy diabetic ward discharging 60 patients
a week would be sending about 17 to the discharge team. That is a
stretch but it is a list a team can look at. At 0.08 it would be
40 a week, which is not a list, it is the whole ward again. If a
real team told me they could manage 10 a week, the honest response
is to raise the threshold to about 0.18 and accept recall falling
to a quarter - not to pretend the model can deliver both.

On missed readmissions: at 0.125 I miss 53% of them. I want to
write that plainly rather than bury it in a recall figure. Just
over half the patients who come back within 30 days were never
flagged. There is no threshold on this model that fixes that
without flagging almost everyone, which is the finding, not a
footnote.

On cost per catch: precision 18.7% means roughly five patients
worked up per readmission found. And found is not prevented -
discharge interventions cut readmission in relative terms, not
absolute, so the number needed to intervene to actually prevent
one is several times higher again. That is the figure an
administrator would ask for and I would rather state it than let
precision stand in for it.

### Two tiers rather than one

Binary triage is not how a ward works, and I do not think one
threshold is the right output here.

What I would actually deploy is a high tier at 0.25 and up -
roughly 5% of discharges, precision near 31% - getting the full
bundle: pharmacist review, structured education, booked follow-up.
Then a middle tier from 0.125 to 0.25 getting the cheap
intervention, a follow-up phone call at 72 hours. Everyone below
0.125 gets standard discharge.

This matters because it changes what precision has to buy. A
wasted phone call costs a few minutes. A wasted pharmacist hour
costs an hour that another patient needed. Matching intervention
intensity to confidence is how you make a model with 18.7%
precision clinically defensible - you spend heavily only where the
model is most sure, and cheaply where it is merely suspicious.

### The Kenyan reframe

The calculus shifts, and not in the direction I first expected.

The obvious argument is that scarcer staff means a false positive
costs more, so the threshold should rise and the flagged list
shrink. That is half right. But the other half is that the
intervention itself is different. At Murang'a there is no
discharge-planning team to ration. What exists is a nurse with a
ward round to finish and ten minutes at the bedside. The
intervention is not a bundle, it is whether someone sits down and
checks that the patient understands the insulin, can afford the
next month of it, and knows when to come back.

That is cheaper than a US bundle, which argues for a lower
threshold, not a higher one. But it is also less effective per
patient, which argues the other way.

The part that does not transfer at all is the outcome. A Kenyan
patient who deteriorates may not readmit - they may not come back
because they cannot afford the fare, or they present somewhere
else entirely with no shared record. The 30-day readmission I am
predicting here is partly a measure of access. Where access is the
binding constraint, the model would be predicting the wrong thing,
and the right target is loss to follow-up. I will take that up
properly in the transferability section.

### What this section leaves me with

At my chosen operating point I flag about one discharge in four,
four out of five of those patients were never going to come back,
and I still miss half the ones who do. That is not a tool anyone
should run unsupervised.

What it is worth is as a ranking. The patients it puts at the top
are roughly twice as likely to readmit as an unranked pick, and if
a team has ten slots, filling them from the top of this list beats
filling them by ward-round impression. That is a real if modest
gain and I am not going to dress it up as more.

---
## 3. Performance by subgroup


In [9]:
# Section 3 — subgroup AUC with bootstrap confidence intervals, so small groups cannot masquerade as findings

MIN_EVENTS = 20          # below this, an AUC is not worth printing
N_BOOT = 500
rng = np.random.default_rng(42)

def auc_ci(yt, yp, n_boot=N_BOOT):
    """Percentile bootstrap CI for AUC. Returns (lo, hi), or (nan, nan) if undefined."""
    n = len(yt)
    boots = []
    for _ in range(n_boot):
        idx = rng.integers(0, n, n)
        ys, ps = yt[idx], yp[idx]
        if 0 < ys.sum() < len(ys):
            boots.append(roc_auc_score(ys, ps))
    if not boots:
        return np.nan, np.nan
    return np.percentile(boots, 2.5), np.percentile(boots, 97.5)

def subgroup_performance(col):
    out = []
    for level, g in preds.groupby(col):
        yt = g["target"].values
        yp = g["prob"].values
        events = int(yt.sum())

        if not (0 < events < len(yt)) or events < MIN_EVENTS:
            auc_str, ci_str = "n/a", f"too few events (n={events})"
        else:
            auc = roc_auc_score(yt, yp)
            lo, hi = auc_ci(yt, yp)
            auc_str = f"{auc:.3f}"
            ci_str = f"{lo:.3f} - {hi:.3f}"

        out.append({
            "group": str(level),
            "n": len(g),
            "events": events,
            "base_rate": round(100 * yt.mean(), 1),
            "auc": auc_str,
            "95% CI": ci_str,
        })
    return pd.DataFrame(out).sort_values("n", ascending=False)

for col in ["age", "race", "gender"]:
    print(f"\n--- {col} ---")
    print(subgroup_performance(col).to_string(index=False))

lo, hi = auc_ci(y, p)
print(f"\nOverall test AUC: {roc_auc_score(y, p):.3f}  [95% CI {lo:.3f} - {hi:.3f}]")


--- age ---
   group    n  events  base_rate   auc                95% CI
 [70-80) 5047     559       11.1 0.646         0.621 - 0.669
 [60-70) 4283     486       11.3 0.641         0.614 - 0.670
 [50-60) 3440     345       10.0 0.673         0.641 - 0.701
 [80-90) 3151     387       12.3 0.626         0.596 - 0.655
 [40-50) 1966     189        9.6 0.677         0.633 - 0.719
 [30-40)  722      69        9.6 0.731         0.663 - 0.794
[90-100)  523      71       13.6 0.596         0.525 - 0.668
 [20-30)  337      61       18.1 0.826         0.773 - 0.870
 [10-20)  138      11        8.0   n/a too few events (n=11)
  [0-10)   27       1        3.7   n/a  too few events (n=1)

--- race ---
          group     n  events  base_rate   auc               95% CI
      Caucasian 14644    1669       11.4 0.658        0.644 - 0.671
AfricanAmerican  3715     394       10.6 0.662        0.634 - 0.691
        Unknown   443      37        8.4 0.619        0.529 - 0.701
       Hispanic   416      35 

In [10]:
# Under-20 discrimination, with the caveat made quantitative

u = preds[preds["under_20"] == True]
print(f"Under-20 group: {len(u):,} encounters, {int(u['target'].sum())} readmissions")

if 0 < u["target"].sum() < len(u):
    auc_u = roc_auc_score(u["target"], u["prob"])
    print(f"AUC: {auc_u:.3f}")
    print(f"\nThis rests on {int(u['target'].sum())} events. The number is reported")
    print("because notebook 01 committed to it, not because it is reliable.")
else:
    print("Too few events in one class to compute an AUC at all.")

Under-20 group: 165 encounters, 12 readmissions
AUC: 0.700

This rests on 12 events. The number is reported
because notebook 01 committed to it, not because it is reliable.


### What the subgroup table tells me

I expected to be looking at race and sex here, because that is
where fairness questions usually land. The most interesting thing
in the table turned out to be age.

Reading the age bands by age rather than by size, the pattern is
hard to miss. In the 30-40 group the model achieves 0.731. By
80-90 it is down to 0.626, and in the over-90s 0.596. And the base
rate moves the opposite way - 9.6% in the 30s, 13.6% in the
over-90s. The model is worst exactly where the risk is highest.

I wanted to know whether that was a real pattern or seven bands of
noise lining up by chance, so I bootstrapped the intervals. The
over-90 interval runs 0.525-0.668 and the 30-40 interval runs
0.663-0.794. They do not overlap. Neither do 80-90 and 30-40. So
this is a genuine separation, not something I am squinting at.

The clinical reading makes sense to me. In a 35-year-old,
readmission is driven by things this dataset actually holds -
prior admissions, poor control, recurrent DKA. In a 90-year-old it
is frailty, polypharmacy, cognition, and whether anyone at home is
managing the insulin. None of that is in the file. So the ceiling
I have been describing is not spread evenly across the ward. It is
concentrated in the elderly, who are most of my admissions.

The 20-30 group is the other end of that story. AUC 0.826, and the
interval 0.773-0.870 sits nowhere near the overall 0.657. They
also carry the highest base rate in the table at 18.1%. My guess
is type 1 diabetics with recurrent DKA, where number_inpatient is
doing real work. It is only 61 events so I would not build
anything on it, but it points the same way: the model does well
where the drivers of readmission are clinical and badly where they
are social.

### Race and sex

Race comes out clean on the two groups large enough to judge.
Caucasian 0.658 (0.644-0.671) against African American 0.662
(0.634-0.691) - the intervals sit almost on top of each other,
across 14,644 and 3,715 encounters. That is a real null and I am
glad to have it, given what notebook 02 turned up about missing
race coding tracking acuity. Hispanic, Other and Unknown I cannot
say anything about; the Other interval runs 0.492-0.699, which
includes chance.

Sex I nearly over-read. Female 0.649 (0.632-0.664), Male 0.666
(0.648-0.684). My first instinct was to call that a small but real
difference, because the point estimates are 1.7 apart on roughly
ten thousand encounters each. But the intervals overlap across
0.648-0.664, and the female estimate of 0.649 sits comfortably
inside the male interval. Only the male estimate falls outside the
female one, and that asymmetry is not evidence of anything - it is
what you get when two overlapping intervals differ slightly in
width. So the honest reading is that I cannot separate these two.
There may be a real gap, it may be zero, and this test does not
tell me which. Section 4 is the one that settles whether it
matters in practice.

### What I am not claiming

Asian, the 10-20 band and the 0-10 band are printed as n/a. Asian
has 7 events, the 0-10 band has one. An AUC computed on a single
event is arithmetic, not a measurement, and I would rather show
nothing than show a number that invites a reader to compare it
with 0.658 on 1,669 events.

One caveat on all of these. I bootstrapped encounters, but I split
the data on patient_nbr, so encounters from the same patient are
correlated and my real intervals are somewhat wider than what is
printed. That does not change the age finding - the gap there is
too large - but it makes the sex comparison even less separable
than it already looks.

Finally, everything in this section is about ranking. AUC asks
whether the model orders patients correctly within a group. It is
completely blind to whether the model's probabilities are set too
high or too low for a group, which is what actually determines who
gets flagged at a fixed threshold. That is section 4's question
and I should not read these nulls as meaning the model treats
these groups equally.ould not read these nulls as meaning the model treats
these groups equally.

---
## 4. Does the chosen threshold treat subgroups equally?

AUC per subgroup asks whether the model *ranks* equally well across groups. But
the model is deployed at a fixed threshold, and a single threshold can still land
very differently on different groups if their risk distributions differ.

This checks recall — the share of each group's readmissions that the threshold
catches. If the threshold catches 60% of readmissions in one group and 30% in
another, that is a fairness problem the overall number would hide.

In [11]:
# Recall and flag rate per subgroup at the chosen threshold

def subgroup_at_threshold(col, thresh=CHOSEN_THRESHOLD):
    out = []
    for level, g in preds.groupby(col):
        gy = g["target"].values
        gp = (g["prob"].values >= thresh).astype(int)
        tp = int(((gp == 1) & (gy == 1)).sum())
        fn = int(((gp == 0) & (gy == 1)).sum())
        flagged = int(gp.sum())
        out.append({
            "group": str(level),
            "n": len(g),
            "flagged_pct": round(100 * flagged / len(g), 1),
            "recall": round(100 * tp / (tp + fn), 1) if (tp + fn) else np.nan,
        })
    return pd.DataFrame(out).sort_values("n", ascending=False)

for col in ["age", "race", "gender"]:
    print(f"\n--- {col} (threshold {CHOSEN_THRESHOLD}) ---")
    print(subgroup_at_threshold(col).to_string(index=False))


--- age (threshold 0.125) ---
   group    n  flagged_pct  recall
 [70-80) 5047         34.1    54.2
 [60-70) 4283         25.7    42.0
 [50-60) 3440         17.6    34.5
 [80-90) 3151         36.5    50.9
 [40-50) 1966         21.6    43.9
 [30-40)  722         27.0    56.5
[90-100)  523         34.8    46.5
 [20-30)  337         29.7    65.6
 [10-20)  138          2.9     9.1
  [0-10)   27          3.7     0.0

--- race (threshold 0.125) ---
          group     n  flagged_pct  recall
      Caucasian 14644         28.3    46.6
AfricanAmerican  3715         29.5    51.8
        Unknown   443         13.1    21.6
       Hispanic   416         27.9    42.9
          Other   299         13.7    24.3
          Asian   117         29.1    71.4

--- gender (threshold 0.125) ---
 group     n  flagged_pct  recall
Female 10594         28.2    45.8
  Male  9040         27.6    48.0


### Is the model fair at this threshold?

**Sex first, because it is the clean one.** Female 28.2% flagged
and 45.8% recall, male 27.6% and 48.0%. Those are close enough
that I would not act on the difference, and it is worth noting
that the small AUC gap I found in section 3 - 0.649 against 0.666 -
does not translate into anything at the threshold. A model can
rank slightly better in one group and still flag both the same
way. So sex is not a fairness problem here.

**Race is fine on the two groups I can actually judge, and a
problem on the two I nearly ignored.** Caucasian 28.3% flagged
and 46.6% recall, African American 29.5% and 51.8%. African
American patients are marginally better served despite a slightly
lower base rate. I have no complaint there.

Then Unknown flags 13.1% with recall 21.6%, and Other flags 13.7%
with recall 24.3%. Every other group sits at 27-30% flagged. These
two are at less than half that, and they catch roughly half as
many of their readmissions.

My first instinct was that this is just the threshold correctly
tracking lower risk - Unknown has a base rate of 8.4%, so of course
fewer get flagged. But that does not hold up. Other has a base rate
of 12.4%, which is higher than Caucasian's 11.4%, and it still
flags at less than half the rate. So the model is not assigning
these patients lower probabilities because they are lower risk. It
is assigning them lower probabilities than their actual outcomes
justify.

That is a calibration failure, and it is specifically the kind
section 3 could not see. AUC only asks whether the ordering within
a group is right. It says nothing about whether the whole group's
probabilities are shifted downward, which is exactly what
determines who clears a fixed threshold.

**Where I think this comes from.** Notebook 02 found that missing
race coding is not random - it tracks acuity. Whatever
circumstances lead to race going unrecorded also correlate with
features the model reads as lower risk. So a data-quality artefact
is producing differential treatment. The patients being
under-served are not defined by a clinical characteristic at all.
They are defined by a gap in the paperwork.

**Age does not behave the way I expected, and the premise of the
question is wrong.** I assumed unequal flagging across age bands
would be fine because the base rates genuinely differ. Checking it,
recall does not track risk:

- 50-60: base rate 10.0%, recall 34.5%
- 40-50: base rate 9.6%, recall 43.9%
- 30-40: base rate 9.6%, recall 56.5%
- 90-100: base rate 13.6%, recall 46.5%

Three bands with near-identical risk and recall ranging from 34.5%
to 56.5%. And the over-90s, the highest-risk band in the table, get
lower recall than the 30-40s, the lowest-risk. So the flags are not
following risk.

What they are following is section 3's finding. Recall tracks how
well the model discriminates within a band, not how sick the band
is. The elderly are flagged plenty - 34.8% of the over-90s - but
because the model cannot sort them, those flags land less
accurately. The differential treatment here is discrimination
quality showing up at the threshold.

**Is any of this large enough to act on, and what would acting
mean.** The Unknown and Other gap is large enough. Half the flag
rate and half the recall is not a rounding difference.

I do not think a per-group threshold is the answer. Setting a lower
threshold for Unknown means assigning patients different clinical
treatment on the basis of whether a clerk filled in a field, which
is worse than the problem. And it does not address the cause -
the probabilities would still be wrong, I would just be papering
over them.

What I would actually do is treat missingness as the signal it
already is. Notebook 02 showed that unrecorded race carries
information; the honest response is to model it explicitly rather
than let it act through the back door, and to check in notebook 05
whether an indicator for missing demographic data changes the
picture.

For age, I would not adjust anything. The gap is caused by the
model being weaker in the elderly, and the fix for that is better
features - frailty, cognition, who is at home - not a different
threshold. Lowering the threshold for the over-90s would flag more
of them without flagging them any more accurately.

**What I would document as a limitation.** That this model
under-flags patients with incomplete demographic records by
roughly half, that the cause is a data-quality artefact rather
than clinical risk, and that any deployment would need monitoring
of flag rates by recording completeness, not just by race
category. A fairness audit that only compared Caucasian against
African American would have passed this model cleanly and missed
it entirely.

---
## 5. Calibration within subgroups

The commitment from notebook 01 was specific: report calibration for subgroups
whose base rate differs from the cohort, because a model calibrated on average can
still be systematically wrong for such a group. The under-20 group at roughly half
the cohort base rate is the test case.

A model that says "20% risk" should see about 20% of those patients readmitted —
within each group, not just overall.

In [13]:
# Mean calibration gap overall

prob_true, prob_pred = calibration_curve(y, p, n_bins=10, strategy="quantile")
gaps = np.abs(prob_true - prob_pred)
print(f"Mean absolute calibration gap across deciles: {gaps.mean():.4f}")
print(f"Largest single-bin gap:                       {gaps.max():.4f}")
print(f"Overall Brier score:                          {brier_score_loss(y, p):.4f}")

Mean absolute calibration gap across deciles: 0.0071
Largest single-bin gap:                       0.0290
Overall Brier score:                          0.0948


In [15]:
# Section 5 — subgroup calibration with Wilson intervals on the observed rate, so small-group gaps can be judged rather than eyeballed

def calibration_by_group(col, bins=(0, 0.08, 0.12, 0.20, 1.0), min_band=30, min_group=200):
    print(f"\n--- {col} ---")
    for level, g in preds.groupby(col):
        if len(g) < min_group:
            continue
        gg = g.copy()
        gg["band"] = pd.cut(gg["prob"], bins=bins)

        rows = []
        for band, b in gg.groupby("band", observed=True):
            if len(b) < min_band:
                continue
            n = len(b)
            events = int(b["target"].sum())
            pred = b["prob"].mean()
            obs = events / n
            lo, hi = proportion_confint(events, n, alpha=0.05, method="wilson")
            # predicted rate falling outside the observed CI = calibration gap unlikely to be chance
            verdict = "" if lo <= pred <= hi else ("  <-- UNDER-predicts" if pred < lo else "  <-- OVER-predicts")
            rows.append(f"{str(band):<16} n={n:>5}  ev={events:>4}  "
                        f"pred {pred:.3f}  obs {obs:.3f}  [{lo:.3f}-{hi:.3f}]{verdict}")

        if rows:
            print(f"  {level}")
            for r in rows:
                print(f"    {r}")

# whole-group check: is the model's average prediction right for this group overall?
def group_calibration_ratio(col, min_group=100):
    out = []
    for level, g in preds.groupby(col):
        if len(g) < min_group:
            continue
        n, events = len(g), int(g["target"].sum())
        pred, obs = g["prob"].mean(), events / n
        lo, hi = proportion_confint(events, n, alpha=0.05, method="wilson")
        out.append({
            "group": str(level), "n": n, "events": events,
            "mean_pred": round(pred, 4), "observed": round(obs, 4),
            "obs_95CI": f"{lo:.3f}-{hi:.3f}",
            "ratio_pred_obs": round(pred / obs, 2) if obs else np.nan,
            "flag": "" if lo <= pred <= hi else ("UNDER" if pred < lo else "OVER"),
        })
    return pd.DataFrame(out).sort_values("n", ascending=False)

for col in ["gender", "race", "age"]:
    calibration_by_group(col)

for col in ["gender", "race", "age"]:
    print(f"\n=== whole-group calibration: {col} ===")
    print(group_calibration_ratio(col).to_string(index=False))


--- gender ---
  Female
    (0.0, 0.08]      n= 3540  ev= 216  pred 0.063  obs 0.061  [0.054-0.069]
    (0.08, 0.12]     n= 3761  ev= 396  pred 0.098  obs 0.105  [0.096-0.116]
    (0.12, 0.2]      n= 2346  ev= 360  pred 0.149  obs 0.153  [0.139-0.169]
    (0.2, 1.0]       n=  947  ev= 227  pred 0.286  obs 0.240  [0.214-0.268]  <-- OVER-predicts
  Male
    (0.0, 0.08]      n= 3004  ev= 164  pred 0.063  obs 0.055  [0.047-0.063]
    (0.08, 0.12]     n= 3264  ev= 321  pred 0.097  obs 0.098  [0.089-0.109]
    (0.12, 0.2]      n= 1993  ev= 273  pred 0.148  obs 0.137  [0.123-0.153]
    (0.2, 1.0]       n=  779  ev= 222  pred 0.299  obs 0.285  [0.254-0.318]

--- race ---
  AfricanAmerican
    (0.0, 0.08]      n= 1167  ev=  57  pred 0.063  obs 0.049  [0.038-0.063]  <-- OVER-predicts
    (0.08, 0.12]     n= 1352  ev= 124  pred 0.098  obs 0.092  [0.077-0.108]
    (0.12, 0.2]      n=  798  ev= 130  pred 0.149  obs 0.163  [0.139-0.190]
    (0.2, 1.0]       n=  398  ev=  83  pred 0.292  obs 0.209  

### What subgroup calibration shows

The overall number first, because it is the one I would have
reported if I had stopped here. Mean absolute calibration gap
across deciles: 0.0071. Largest single-bin gap: 0.029. Brier score
0.0948. On those figures this model is beautifully calibrated and
I would have written a satisfied paragraph and moved on.

Underneath that number, one group is being under-scored by about
30%, another over-scored by 12%, and the top risk band - the one I
proposed building a high-intensity tier on - is unreliable almost
everywhere. None of that is visible in the overall figure, because
the errors run in opposite directions and cancel. That is the
whole argument for doing this section, and it is the most useful
thing I have found in this notebook.

### Other is under-predicted

Whole-group mean prediction 0.0872 against an observed rate of
0.1237, a ratio of 0.70, and the prediction sits below the 95%
interval on the observed rate. The model scores this group about
30% lower than their outcomes justify.

What convinced me is not any single band - each band's interval
contains its own prediction - but that all three drift the same
way: 0.058 against 0.086, 0.097 against 0.154, 0.144 against
0.200. Three bands leaning in one direction is what the aggregate
is picking up. The group also has nobody at all above 0.20, on 299
encounters where the cohort rate would put roughly fifteen there.
The model never places an Other patient in the high-risk band.

This is what was driving the section 4 finding. Other flags at
13.7% while everyone else sits near 28%, and it is not because
they are lower risk - their base rate of 12.4% is higher than
Caucasian's 11.4%. They are being scored too low, so they do not
clear the threshold.

I cannot say why. I expected this to be about missing demographic
coding, but Unknown comes back at a ratio of 1.03 with no flag at
all, so that explanation does not hold. Unknown's low flag rate is
the model correctly reading a genuinely lower-risk group. Other is
something else and I do not have a mechanism for it. On 299
patients and 37 events I would call this suggestive rather than
established, and something to check in a larger sample before
acting on.

### African American patients are over-predicted at the top

Whole-group ratio 1.12, flagged. It is concentrated where it
matters: in the top band the model predicts 0.292 and observes
0.209, interval 0.172-0.251, on 398 patients. Caucasian patients
in the same band come out at 0.292 against 0.280, almost exact.

So the ranking null I found in section 3 - AUC 0.662 against 0.658
- was real but incomplete. The model orders these patients as well
as anyone else and then sets their probabilities too high at the
top end.

### The top band is unreliable almost everywhere

This is the finding I did not go looking for. In the (0.2, 1.0]
band: Female 0.286 against 0.240, African American 0.292 against
0.209, 70-80 0.285 against 0.229, 50-60 0.287 against 0.243,
90-100 0.298 against 0.208. Three of those are flagged and the
rest fall short of significance while pointing the same way.

That is a problem for the two-tier design I proposed in section 2.
I wanted a high tier at 0.25 and above getting the expensive
intervention, on the reasoning that precision is best where the
model is most confident. But confidence is exactly where it is
least accurate. The patients I would be spending pharmacist hours
on are scored higher than their outcomes support, in most groups I
can measure. I need to revisit that tier boundary rather than
carry it forward as designed.

### Young adults are scored too low

The 20-30 group has the highest base rate in the cohort at 18.1%
and a whole-group ratio of 0.82. In the 0.12-0.2 band the model
predicts 0.147 and observes 0.354, interval 0.234-0.496 - the
largest gap anywhere in this table.

Set alongside their AUC of 0.826, the picture is consistent. The
model ranks young adults better than any other group and then
refuses to believe how sick they are. If these are type 1
diabetics with recurrent DKA, the ordering signal is there and the
absolute level is not.

### What I got wrong

Notebook 01 predicted that an adult-fit model would over-predict in
children, because their base rate is roughly half the cohort's.
The 10-20 band comes back at a ratio of 0.60 - under-prediction,
the opposite direction. It rests on 11 events and is not
significant, so I am not claiming the reverse either. But the
prediction was wrong and I would rather record that than drop it
quietly.

### The caveat on all of this

The intervals are Wilson intervals on the observed rate within
each band, which handles small samples honestly. What they do not
handle is that my split was grouped on patient_nbr, so encounters
from the same patient are correlated and the true intervals are
somewhat wider. The Other and 20-30 findings are the ones most
exposed to that, since they rest on the fewest events. The 70-80
over-prediction, on 5,047 encounters, is the most solid thing
here even though it is the least dramatic.

---
## 6. Would using the model beat not using it?



In [14]:
# Net comparison at the chosen threshold

flag = (p >= CHOSEN_THRESHOLD).astype(int)
caught = int(((flag == 1) & (y == 1)).sum())
total_events = int(y.sum())
reviewed = int(flag.sum())

print("Treat nobody:")
print(f"  Reviews: 0    Readmissions caught in time: 0 of {total_events}")
print("\nTreat everybody:")
print(f"  Reviews: {len(y):,}    Readmissions caught: {total_events} of {total_events}")
print(f"\nModel at threshold {CHOSEN_THRESHOLD}:")
print(f"  Reviews: {reviewed:,} ({100*reviewed/len(y):.0f}% of discharges)")
print(f"  Readmissions caught: {caught} of {total_events} ({100*caught/total_events:.0f}%)")
print(f"  Reviews per readmission caught: {reviewed/caught:.1f}")

Treat nobody:
  Reviews: 0    Readmissions caught in time: 0 of 2179

Treat everybody:
  Reviews: 19,634    Readmissions caught: 2179 of 2179

Model at threshold 0.125:
  Reviews: 5,486 (28% of discharges)
  Readmissions caught: 1019 of 2179 (47%)
  Reviews per readmission caught: 5.4


In [22]:
# Section 6 — compare the model against a one-variable clinical rule at the same review budget
cohort = pd.read_csv(f"{DATA_DIR}/processed/test_predictors.csv")

check = preds.merge(cohort, on="encounter_id", how="left", validate="one_to_one")
assert len(check) == len(preds), "join changed row count"
assert check["number_inpatient"].isna().sum() == 0, "some encounters did not match"
print(f"Joined cleanly: {len(check):,} rows\n")

BUDGET = int((p >= CHOSEN_THRESHOLD).sum())
total_events = int(y.sum())
y_arr = check["target"].values

def catches_at_budget(scores, label):
    """Take the top BUDGET patients by score; count readmissions among them."""
    order = np.argsort(-np.asarray(scores), kind="stable")
    caught = int(y_arr[order[:BUDGET]].sum())
    print(f"{label:<42} caught {caught:>5} of {total_events} ({100*caught/total_events:.1f}%)")
    return caught

print(f"Review budget fixed at {BUDGET:,} patients ({100*BUDGET/len(check):.0f}% of discharges)\n")

# random tie-break so patients with equal counts are not ordered by row position
rng_tb = np.random.default_rng(0)
jitter = rng_tb.random(len(check)) * 1e-6

m = catches_at_budget(check["prob"].values, "Model (37 features)")
s = catches_at_budget(check["number_inpatient"].values + jitter, "Prior inpatient admissions alone")
e = catches_at_budget(check["number_inpatient"].values + check["number_emergency"].values + jitter,
                      "Prior inpatient + emergency visits")
r = catches_at_budget(rng_tb.random(len(check)), "Random selection")

print(f"\nModel over the one-variable rule: {m - s:+d} readmissions "
      f"({100*(m-s)/total_events:+.1f} pp of recall)")
print(f"Model over random:                {m - r:+d} readmissions "
      f"({100*(m-r)/total_events:+.1f} pp of recall)")

Joined cleanly: 19,634 rows

Review budget fixed at 5,486 patients (28% of discharges)

Model (37 features)                        caught  1019 of 2179 (46.8%)
Prior inpatient admissions alone           caught   972 of 2179 (44.6%)
Prior inpatient + emergency visits         caught   967 of 2179 (44.4%)
Random selection                           caught   567 of 2179 (26.0%)

Model over the one-variable rule: +47 readmissions (+2.2 pp of recall)
Model over random:                +452 readmissions (+20.7 pp of recall)


In [21]:
# Write a small predictor extract so section 6 runs from a clone without the 14MB cohort file
cohort_full = pd.read_csv(f"{DATA_DIR}/processed/cohort.csv",
                          usecols=["encounter_id", "number_inpatient", "number_emergency"])
cohort_full.to_csv(f"{DATA_DIR}/processed/test_predictors.csv", index=False)
print(f"Wrote {len(cohort_full):,} rows to test_predictors.csv")

Wrote 99,316 rows to test_predictors.csv


### Would using the model beat not using it?

The notebook set this up against two comparators, treat nobody and
treat everybody, and against those two the model looks good. At my
threshold I review 5,486 patients, 28% of discharges, and catch
1,019 of 2,179 readmissions - 47%. Treating nobody catches none.
Treating everybody catches all of them but means reviewing 19,634
discharges, which no service does. Reviewing 28% to catch 47% is
better than proportional, and 5.4 reviews per readmission found.

The problem is that neither comparator is real. Nobody treats
everybody, and a ward with no model does not pick patients at
random either. So I added the comparison that actually matters:
what if you just sorted by how many times the patient has been
admitted in the past year, and reviewed the same 5,486?

That catches 972. My model catches 1,019.

Forty-seven readmissions. 2.2 percentage points of recall. Three
notebooks of cleaning, thirty-seven features, diagnosis grouping,
medication encoding, and the whole thing is 95% reproduced by a
single integer a registrar can read off the notes in five seconds.
Adding prior emergency visits to the rule makes it slightly worse,
967, so the extra signal there is noise at this budget.

Against random selection the model does catch 452 more
readmissions, 20.7 points of recall, so it is certainly doing
something. But random is not what happens on a ward. What happens
on a ward is that someone looks at the notes, sees four admissions
this year, and thinks this one will be back. That is the
comparator, and I beat it by 2.2 points.

### What I think this actually means

This is the clearest evidence I have that the ceiling is in the
data rather than the model. I have been arguing that for a while
on the basis of what is missing - no vitals, no renal function, no
social context. This shows it from the other end: almost all the
predictive signal in thirty-seven features lives in one of them.
There is nothing left for a better algorithm to find. Whatever
gradient boosting gives me in notebook 05, it is not going to find
signal that is not there.

### So would I deploy it?

Against nothing, yes. Against random, clearly. Against a clinician
sorting by prior admissions, the honest answer is barely - and I
have to weigh those 47 patients against what a deployed model
actually costs: monitoring, retraining, calibration audits, a
fairness review, and someone owning it. A service manager could
reasonably look at 2.2 percentage points and decline. I would not
argue with them.

There is one thing the model has that the heuristic does not, and
it is not accuracy. It produces a calibrated probability rather
than a rank. That means I can set a threshold on it, tier
interventions against it, and audit it - which is exactly what
sections 4 and 5 did, and it is how I found that Other is
under-scored by 30% and that the top band over-predicts nearly
everywhere. You cannot run a calibration audit on "sort by prior
admissions". You cannot check whether it under-serves a subgroup,
because it does not produce a number to check.

So the argument for this model is a governance argument, not a
clinical one. It makes the ranking explicit, inspectable and
challengeable, where the informal version is none of those things
and may well carry the same biases invisibly. I think that is
worth something. But I am not going to write it up as though the
2.2 points were the point.

### One number I should have printed

Catching is not preventing. Discharge interventions reduce
readmission in relative terms, somewhere in the region of 20-30%,
so of the 1,019 readmissions I flag, an intervention might prevent
perhaps 200-300. Against 2,179 total, that is under 15% of the
problem, for 5,486 reviews. Roughly 20 to 27 reviews per
readmission actually prevented. That is the figure a clinical
director would want and it is the one I would lead with.

---
## 7. Summary

I set the threshold at 0.125, the point where F1 peaks. It flags
27.9% of discharges - 5,486 of 19,634 encounters - and catches
1,019 of 2,179 readmissions, which is 47%. That commits a
discharge team to reviewing roughly one patient in four, working
up about 5.4 patients for every readmission found, and still
missing 1,160 of them. I want that last number written plainly
rather than folded into a recall figure: more than half the
patients who come back within 30 days were never flagged. There is
no threshold on this model that fixes that without flagging almost
everyone, and I tested that directly - weighting recall with
F-beta pushed the model to flag 86% of discharges at a precision
barely above the base rate, which is not a risk model, it is a
rubber stamp.

On fairness, the headline number is misleading and that is the
finding. Overall calibration looks excellent, a mean absolute
decile gap of 0.0071. Underneath it, patients coded race "Other"
are scored about 30% below their observed rate and never place
above 0.20 at all, which is why they flag at 13.7% while everyone
else sits near 28% - and their base rate is higher than
Caucasian's, so this is not the threshold tracking lower risk.
African American patients are over-predicted at the top band,
0.292 against 0.209 observed. The top band over-predicts in almost
every group I can measure, which is a direct problem for the
tiered design I proposed, because that is where the expensive
intervention would sit. Sex is clean at the threshold - 28.2%
against 27.6% flagged, recall within two points - and the small
AUC difference I found in section 3 is not separable once the
intervals are taken seriously. Race is clean on the two groups
large enough to judge. Age is where the model is weakest:
discrimination falls monotonically as age rises, from 0.731 in the
30s to 0.596 in the over-90s with non-overlapping intervals, so
the model performs worst exactly where the risk is highest. I
would document the "Other" under-scoring and the top-band
over-prediction as deployment limitations requiring monitoring,
and I would note that a fairness audit comparing only Caucasian
against African American would have passed this model cleanly and
missed both.

On deployment, the two comparators the notebook set up are not the
real ones. Against treating nobody the model obviously wins;
against treating everybody it wins on workload. But no ward does
either. What a ward does is look at how many times the patient has
been in recently - so I ranked by prior inpatient admissions alone
at the same review budget. That catches 972. The model catches
1,019. Forty-seven readmissions, 2.2 percentage points, for
thirty-seven features and three notebooks of work. Against random
selection the model finds 452 more, so it is doing something real,
but random is not the alternative. I would not put this in front
of a discharge team as an accuracy improvement, because it barely
is one. What I would argue is narrower: the model produces a
calibrated probability rather than a rank, which is what let me
threshold it, tier it, and audit it - and auditing is how I found
the "Other" gap. You cannot run a subgroup calibration check on
"sort by prior admissions", which means the informal version may
carry the same bias invisibly. That is a governance argument
rather than a clinical one, and I would rather make it honestly
than dress up 2.2 points as patient benefit.

---

**Next:** `05_model_comparison.ipynb` — gradient boosting and a
reduced five-feature model against this baseline, each judged on
the same fixed-budget comparator rather than on AUC. The question
is specific: does any model beat prior inpatient admissions alone
by more than the 2.2 percentage points this one manages? If none
of them does, the ceiling is in the data and I can say so with
evidence rather than assertion.

Then `06_explainability.ipynb` — SHAP on whichever model wins,
risk tiers revisited in light of the top-band over-prediction
found in section 5, the clinician-facing summary, and the Kenya
transferability section promised since notebook 01.cian-facing summary, and the Kenya
transferability section promised since notebook 01.